# Elliptic++ transaction classification with XGBoost

Train an illicit (positive class) versus licit classifier. Download `txs_features.csv` and `txs_classes.csv` from the [Elliptic++ repository](https://github.com/git-disl/EllipticPlusPlus) using its linked Google Drive data, then set `DATA_DIR` below. The repository documents 203,769 transactions, 49 time steps, 183 features.

The attached paper evaluates the original Elliptic dataset, not Elliptic++. Table 1 (PDF p. 11) reports XGBoost AF illicit precision 0.921, recall 0.732, F1 **0.815**. It does not report XGBoost accuracy. F1 comparisons here are cross-dataset and protocol-dependent.

Li, Jia & Su (2025), *Scientific Reports* 15, 22668, [DOI](https://doi.org/10.1038/s41598-025-08365-9).

## 1. Setup

Import libraries, set a random seed, and point DATA_DIR to the folder containing the downloaded CSV files.

### Imports (part 1 of 4)

Run cells in order. This cell continues the 1. setup step.

In [ ]:
from pathlib import Path
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
 confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
 average_precision_score, precision_recall_curve)


### 1. Setup (part 2 of 4)

Run cells in order. This cell continues the 1. setup step.

In [ ]:
from xgboost import XGBClassifier

SEED=42
DATA_DIR=Path("Transactions Dataset")  # change to downloaded CSV directory


### 1. Setup (part 3 of 4)

Run cells in order. This cell continues the 1. setup step.

In [ ]:
FEATURES_FILE=DATA_DIR/"txs_features.csv"
CLASSES_FILE=DATA_DIR/"txs_classes.csv"
PAPER_XGB_F1=0.815  # original Elliptic, paper Table 1, p. 11


### 1. Setup (part 4 of 4)

Run cells in order. This cell continues the 1. setup step.

In [ ]:
for p in (FEATURES_FILE,CLASSES_FILE):
    if not p.exists(): raise FileNotFoundError(f"Missing {p}. Download the CSVs through the repository README and update DATA_DIR.")


## 2. Load and validate files

Inspect the data, identify ID/label columns, check duplicate IDs, and join features to labels.

### Load and preview CSV files (part 1 of 4)

Run cells in order. This cell continues the 2. load and validate files step.

In [ ]:
features,classes=pd.read_csv(FEATURES_FILE),pd.read_csv(CLASSES_FILE)
display(features.head(),classes.head())
def find_col(df,names):
    norm={str(c).lower().replace("_","").replace(" ",""):c for c in df.columns}
    for n in names:
        k=n.lower().replace("_","").replace(" ","")
        if k in norm:return norm[k]
    raise ValueError(f"Expected one of {names}; columns={df.columns.tolist()}")


### Find key columns (part 2 of 4)

Run cells in order. This cell continues the 2. load and validate files step.

In [ ]:
fid=find_col(features,["txId","transactionId","id"])
cid=find_col(classes,["txId","transactionId","id"])
lab=find_col(classes,["class","label","target"])


### Validate transaction IDs (part 3 of 4)

Run cells in order. This cell continues the 2. load and validate files step.

In [ ]:
for df,key in ((features,fid),(classes,cid)):
    if df[key].isna().any() or df[key].astype(str).str.strip().duplicated().any():
        raise ValueError(f"Duplicate or missing IDs in {key}")
features=features.copy(); classes=classes.copy()
features["_id"]=features[fid].astype(str).str.strip()


### 2. Load and validate files (part 4 of 4)

Run cells in order. This cell continues the 2. load and validate files step.

In [ ]:
classes["_id"]=classes[cid].astype(str).str.strip()
print("Unmatched feature IDs:",len(set(features._id)-set(classes._id)),
      "unmatched class IDs:",len(set(classes._id)-set(features._id)))
data=features.merge(classes[["_id",lab]],on="_id",validate="one_to_one").rename(columns={lab:"_raw"})


## 3. Map labels and inspect class balance

Map documented labels, exclude unknowns from supervised work, and visualize the known class distribution.

### Map classes and inspect counts (part 1 of 3)

Run cells in order. This cell continues the 3. map labels and inspect class balance step.

In [ ]:
raw=data["_raw"].astype(str).str.strip().str.lower()
data["_y"]=raw.map({"1":1,"illicit":1,"2":0,"licit":0,"3":np.nan,
                    "unknown":np.nan,"unlabeled":np.nan,"unlabelled":np.nan})
print("Raw class counts:",raw.value_counts(dropna=False).to_dict())


### 3. Map labels and inspect class balance (part 2 of 3)

Run cells in order. This cell continues the 3. map labels and inspect class balance step.

In [ ]:
print("Unrecognized classes:",sorted(raw[data._y.isna() & ~raw.isin(["3","unknown","unlabeled","unlabelled"])].unique()))
display(data["_y"].value_counts(dropna=False).rename("rows").to_frame())
known=data.loc[data._y.notna()].copy(); known["_y"]=known["_y"].astype(int)


### 3. Map labels and inspect class balance (part 3 of 3)

Run cells in order. This cell continues the 3. map labels and inspect class balance step.

In [ ]:
if known["_y"].nunique()!=2: raise ValueError("Both licit and illicit labels are needed.")
known["_y"].map({0:"Licit",1:"Illicit"}).value_counts().plot.bar(title="Known class distribution")
plt.ylabel("Transactions"); plt.show()


## 4. Prepare predictors and create chronological splits

Keep identifiers and labels out of the predictors. The latest time steps are reserved for the final test set.

### Choose time column (part 1 of 7)

Run cells in order. This cell continues the 4. prepare predictors and create chronological splits step.

In [ ]:
time_col=find_col(known,["Time step","time_step","timestep"])
exclude={fid,cid,"_id","_raw","_y"}
feature_cols=[c for c in known.columns if c not in exclude]


### 4. Prepare predictors and create chronological splits (part 2 of 7)

Run cells in order. This cell continues the 4. prepare predictors and create chronological splits step.

In [ ]:
X=known[feature_cols].copy()
for c in X: X[c]=pd.to_numeric(X[c],errors="coerce")
X=X.replace([np.inf,-np.inf],np.nan).dropna(axis=1,how="all")


### Set labels and time values (part 3 of 7)

Run cells in order. This cell continues the 4. prepare predictors and create chronological splits step.

In [ ]:
y=known["_y"]
times=pd.to_numeric(known[time_col],errors="coerce")
if times.isna().any(): raise ValueError("Invalid time steps; cannot split chronologically.")


### Choose chronological periods (part 4 of 7)

Run cells in order. This cell continues the 4. prepare predictors and create chronological splits step.

In [ ]:
steps=np.sort(times.unique())
if len(steps)<3: raise ValueError("Need at least 3 labeled time steps.")
nt,nv=max(1,int(np.ceil(.20*len(steps)))),max(1,int(np.ceil(.15*len(steps))))


### 4. Prepare predictors and create chronological splits (part 5 of 7)

Run cells in order. This cell continues the 4. prepare predictors and create chronological splits step.

In [ ]:
if nt+nv>=len(steps):nt,nv=1,1
tr_steps,va_steps,te_steps=steps[:-(nt+nv)],steps[-(nt+nv):-nt],steps[-nt:]
tr,va,te=times.isin(tr_steps),times.isin(va_steps),times.isin(te_steps)


### 4. Prepare predictors and create chronological splits (part 6 of 7)

Run cells in order. This cell continues the 4. prepare predictors and create chronological splits step.

In [ ]:
Xtr,ytr=X.loc[tr],y.loc[tr]; Xva,yva=X.loc[va],y.loc[va]; Xte,yte=X.loc[te],y.loc[te]
print("Train steps/rows",tr_steps[[0,-1]],len(ytr),"Validation",va_steps[[0,-1]],len(yva),"Test",te_steps[[0,-1]],len(yte))
display(pd.DataFrame({"train":ytr.value_counts(),"validation":yva.value_counts(),"test":yte.value_counts()}).fillna(0).astype(int))


### 4. Prepare predictors and create chronological splits (part 7 of 7)

Run cells in order. This cell continues the 4. prepare predictors and create chronological splits step.

In [ ]:
if ytr.nunique()<2 or yva.nunique()<2:warnings.warn("One class in train/validation; metrics or tuning may be limited.")


## 5. Train baseline and tune XGBoost

Compute imbalance weight from training data, compare a small parameter set on validation F1, and choose a threshold using validation predictions only.

### Compute class weight (part 1 of 5)

Run cells in order. This cell continues the 5. train baseline and tune xgboost step.

In [ ]:
if ytr.sum()==0: raise ValueError("No illicit training samples.")
spw=(ytr==0).sum()/max((ytr==1).sum(),1)
def build(params):
    return XGBClassifier(objective="binary:logistic",eval_metric="logloss",tree_method="hist",
                         random_state=SEED,n_jobs=-1,scale_pos_weight=float(spw),**params)


### Train baseline (part 2 of 5)

Run cells in order. This cell continues the 5. train baseline and tune xgboost step.

In [ ]:
baseline=build({"n_estimators":300,"max_depth":6,"learning_rate":.1}).fit(Xtr,ytr)
params=[
 {"n_estimators":400,"max_depth":3,"learning_rate":.05,"subsample":.8,"colsample_bytree":.8,"min_child_weight":3,"reg_lambda":2},
 {"n_estimators":500,"max_depth":4,"learning_rate":.03,"subsample":.8,"colsample_bytree":.9,"min_child_weight":5,"reg_lambda":3},
 {"n_estimators":300,"max_depth":5,"learning_rate":.05,"subsample":.9,"colsample_bytree":.8,"min_child_weight":5,"reg_lambda":2}]
models=[build(p).fit(Xtr,ytr) for p in params]


### 5. Train baseline and tune XGBoost (part 3 of 5)

Run cells in order. This cell continues the 5. train baseline and tune xgboost step.

In [ ]:
scores=[f1_score(yva,(m.predict_proba(Xva)[:,1]>=.5).astype(int),zero_division=0) for m in models]
display(pd.DataFrame({"candidate":range(1,len(models)+1),"validation_F1_at_0.5":scores,"parameters":params}))
tuned=models[int(np.argmax(scores))]


### 5. Train baseline and tune XGBoost (part 4 of 5)

Run cells in order. This cell continues the 5. train baseline and tune xgboost step.

In [ ]:
vprob=tuned.predict_proba(Xva)[:,1]
pr,rc,thresholds=precision_recall_curve(yva,vprob)
if len(thresholds):
    fs=2*pr[:-1]*rc[:-1]/(pr[:-1]+rc[:-1]+1e-12)
    threshold=float(thresholds[int(np.nanargmax(fs))])
else:threshold=.5


### 5. Train baseline and tune XGBoost (part 5 of 5)

Run cells in order. This cell continues the 5. train baseline and tune xgboost step.

In [ ]:
print("Validation-selected threshold",threshold,"validation F1",f1_score(yva,(vprob>=threshold).astype(int),zero_division=0))


## 6. Evaluate once on the held-out test set

Compare baseline and tuned models, print classification metrics, and plot the confusion matrix and precision-recall curve.

### Define evaluation helper (part 1 of 5)

Run cells in order. This cell continues the 6. evaluate once on the held-out test set step.

In [ ]:
def evaluate(name,model,cut):
    prob=model.predict_proba(Xte)[:,1]; pred=(prob>=cut).astype(int)
    return {"model":name,"threshold":cut,"accuracy":accuracy_score(yte,pred),
      "balanced_accuracy":balanced_accuracy_score(yte,pred),"macro_f1":f1_score(yte,pred,average="macro",zero_division=0),
      "illicit_precision":precision_score(yte,pred,zero_division=0),"illicit_recall":recall_score(yte,pred,zero_division=0),
      "illicit_f1":f1_score(yte,pred,zero_division=0),"roc_auc":roc_auc_score(yte,prob) if yte.nunique()==2 else np.nan,
      "pr_auc":average_precision_score(yte,prob) if yte.nunique()==2 else np.nan,"prob":prob,"pred":pred}
base=evaluate("Baseline XGBoost",baseline,.5); result=evaluate("Tuned XGBoost",tuned,threshold)
metric_keys=["model","threshold","accuracy","balanced_accuracy","macro_f1","illicit_precision","illicit_recall","illicit_f1","roc_auc","pr_auc"]


### 6. Evaluate once on the held-out test set (part 2 of 5)

Run cells in order. This cell continues the 6. evaluate once on the held-out test set step.

In [ ]:
display(pd.DataFrame([{k:v for k,v in z.items() if k in metric_keys} for z in (base,result)]).round(4))
print("Positive class = illicit (1)\n",classification_report(yte,result["pred"],labels=[0,1],target_names=["licit","illicit"],zero_division=0))
fig,ax=plt.subplots(1,2,figsize=(11,4))


### 6. Evaluate once on the held-out test set (part 3 of 5)

Run cells in order. This cell continues the 6. evaluate once on the held-out test set step.

In [ ]:
cm=confusion_matrix(yte,result["pred"],labels=[0,1]);ax[0].imshow(cm,cmap="Blues")
ax[0].set(title="Test confusion matrix",xlabel="Predicted",ylabel="Actual")
ax[0].set_xticks([0,1],["Licit","Illicit"]);ax[0].set_yticks([0,1],["Licit","Illicit"])


### 6. Evaluate once on the held-out test set (part 4 of 5)

Run cells in order. This cell continues the 6. evaluate once on the held-out test set step.

In [ ]:
for (i,j),v in np.ndenumerate(cm):ax[0].text(j,i,str(v),ha="center",va="center")
if yte.nunique()==2:
    pp,rr,_=precision_recall_curve(yte,result["prob"]);ax[1].plot(rr,pp,label=f"PR-AUC={result['pr_auc']:.3f}");ax[1].legend()
else:ax[1].text(.5,.5,"Test set has one class",ha="center")
ax[1].set(title="Test precision-recall",xlabel="Recall",ylabel="Precision")


### 6. Evaluate once on the held-out test set (part 5 of 5)

Run cells in order. This cell continues the 6. evaluate once on the held-out test set step.

In [ ]:
plt.tight_layout();plt.show()


## 7. Inspect feature importance and example predictions

Gain importance indicates tree split utility, not causality. The inference example uses held-out rows.

### Feature importance (part 1 of 3)

Run cells in order. This cell continues the 7. inspect feature importance and example predictions step.

In [ ]:
gain=pd.Series(tuned.get_booster().get_score(importance_type="gain"),dtype=float).sort_values(ascending=False).head(20)
mapped={}
for k,v in gain.items():
    ix=int(k[1:]) if k.startswith("f") and k[1:].isdigit() else -1
    mapped[Xtr.columns[ix] if 0<=ix<len(Xtr.columns) else k]=v


### 7. Inspect feature importance and example predictions (part 2 of 3)

Run cells in order. This cell continues the 7. inspect feature importance and example predictions step.

In [ ]:
if mapped:
    pd.Series(mapped).sort_values().plot.barh(figsize=(8,6),title="Top features by gain (not causal)")
    plt.xlabel("Gain");plt.tight_layout();plt.show()
# Inference example using held-out rows; no separate model file is created.
example=Xte.head(10).copy()
example["illicit_probability"]=tuned.predict_proba(example)[:,1]


### 7. Inspect feature importance and example predictions (part 3 of 3)

Run cells in order. This cell continues the 7. inspect feature importance and example predictions step.

In [ ]:
example["prediction"]=np.where(example.illicit_probability>=threshold,"illicit","licit")
display(example[["illicit_probability","prediction"]])


## 8. Results summary

Report measured F1 and accuracy. Paper F1 is a cross-dataset reference; paper accuracy is not reported.

### Results summary (part 1 of 2)

Run cells in order. This cell continues the 8. results summary step.

In [ ]:
f1=result["illicit_f1"];acc=result["accuracy"]
print(f"Elliptic++ tuned illicit F1={f1:.4f}; accuracy={acc:.4f}")
print(f"Paper XGBoost illicit F1 reference={PAPER_XGB_F1:.3f}")


### 8. Results summary (part 2 of 2)

Run cells in order. This cell continues the 8. results summary step.

In [ ]:
print("F1 target status:",("measured above the paper value" if f1>PAPER_XGB_F1 else "not above the paper value") if np.isfinite(f1) else "cannot determine")
print("Accuracy target status: cannot fairly determine; the paper reports no XGBoost accuracy.")
print("F1 is cross-dataset; dataset and evaluation protocol differ.")
